In [ ]:
import time

import torch
from torch import nn

from pract11.Guia11Reformat.src.ClassifierMLP import ClassifierMLP
from pract11.Guia11Reformat.src.Config import Config
from pract11.Guia11Reformat.src.FashionMNISTData import FashionMNISTData
from pract11.Guia11Reformat.src.Optimizer import create_optimizer
from pract11.Guia11Reformat.src.Trainer import Trainer
from pract11.Guia11Reformat.src.Visualizer import Visualizer


def runExperiment(config):
    # Configuración
    device = config.setup()
    # Datos
    data = FashionMNISTData(
        batch_size=config.BATCH_SIZE,
        seed=config.SEED, color=config.COLOR, show_dataset_info=config.show_dataset_info,
    )

    if config.show_sample:
        data.inspect_example()

    # Modelo
    torch.manual_seed(config.SEED)

    model = ClassifierMLP(
        p_dropout=config.DROPOUT_P
    ).to(device)

    # Comprobar dimensiones
    if config.show_dimentions:
        images, labels = next(iter(data.train_loader))

        with torch.no_grad():
            classifications = model(images.to(device))

        print(f"images.shape: {images.shape}")
        print(f"labels.shape: {labels.shape}")

        n_params = sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )
        print(f"Parámetros entrenables: {n_params:,}")

        print("\nDimensiones:")
        print("Entrada:", images.shape)
        print("Logits:", classifications.shape)

    # Función de pérdida y optimizador
    loss_fn = nn.CrossEntropyLoss()

    optimizer = create_optimizer(
        model=model,
        optimizer_name=config.OPTIMIZER_NAME,
        adam_lr=config.ADAM_LR,
        sgd_lr=config.SGD_LR
    )

    print("\nOptimizador:", config.OPTIMIZER_NAME)
    print("Dropout:", config.DROPOUT_P)

    # Untrained
    if config.show_untrained:
        Visualizer.showPredictions(model, device, data)

    # Entrenamiento
    trainer = Trainer(
        model,
        optimizer,
        loss_fn,
        device,
        config
    )
    start = time.perf_counter()
    trainer.fit(
        data.train_loader,
        data.valid_loader,
        epochs=config.EPOCHS
    )

    print(f"Training time: {time.perf_counter() - start:.2f} seconds")
    # Gráfico de pérdidas
    if config.show_loss: Visualizer.plot_history(trainer.history)

    # Evaluación final
    _, _, confusion_matrix = trainer.test(data.test_loader)

    #Trained
    if config.show_trained:
        Visualizer.showPredictions(model, device, data)
    return trainer, data, confusion_matrix


# ============================================================
# EJECUCIÓN
# ============================================================
# showSample=False, showDimentions=False, showUntrained=False, showLoss=False, showTrained=True, showDimentions2=False, showEpochData=True
# runExperiment(Config(dropout_p=0.2, epochs=2)

expies_1 = {
    "SGD_LR 0.001": Config(sgd_lr=0.001, dropout_p=0.2, epochs=30, optimizer_name="SGD", show_sample=False,
                           show_dimentions=False,
                           show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                           show_epoch_data=True, show_init=True, show_dataset_info=False),
    "SGD_LR 0.01": Config(sgd_lr=0.01, dropout_p=0.2, epochs=30, optimizer_name="SGD", show_sample=False,
                          show_dimentions=False,
                          show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                          show_epoch_data=True, show_init=True, show_dataset_info=False),
    "SGD_LR 0.1": Config(sgd_lr=0.1, dropout_p=0.2, epochs=30, optimizer_name="SGD", show_sample=False,
                         show_dimentions=False,
                         show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                         show_epoch_data=True, show_init=True, show_dataset_info=False),
    "Adam_LR 0.0001": Config(adam_lr=0.0001, dropout_p=0.2, epochs=30, optimizer_name="Adam", show_sample=False,
                             show_dimentions=False,
                             show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                             show_epoch_data=True, show_init=True, show_dataset_info=False),
    "Adam_LR 0.001": Config(adam_lr=0.001, dropout_p=0.2, epochs=30, optimizer_name="Adam", show_sample=False,
                            show_dimentions=False,
                            show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                            show_epoch_data=True, show_init=True, show_dataset_info=False),
    "Adam_LR 0.01": Config(adam_lr=0.01, dropout_p=0.2, epochs=30, optimizer_name="Adam", show_sample=False,
                           show_dimentions=False,
                           show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                           show_epoch_data=True, show_init=True, show_dataset_info=False)
}


def pre_run(expies):
    results_together = {}
    for title, exp in expies.items():
        results = {}
        trainer, data, confusion_matrix = runExperiment(exp)
        results[title] = {"trainer": trainer, "data": data, "confusion_matrix": confusion_matrix}
        results_together[title] = {"trainer": trainer, "data": data, "confusion_matrix": confusion_matrix}
        Visualizer.showExpieResults(results)
    print("\n-----------------------------------Results TOGETHER:-----------------------------------\n")
    Visualizer.showExpieResults(results_together)


pre_run(expies_1)



========== CONFIGURATION ==========
Seed:             42
Batch size:       100
Epochs:           30
Dropout:          0.2
Optimizer:        SGD
Adam_LR:        0.001
SGD_LR:        0.001

========== ENVIRONMENT ==========
PyTorch: 2.6.0+cu124
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


Optimizador: SGD
Dropout: 0.2

Iniciando entrenamiento... Opti -> SGD | Drop -> 0.2 | Batch size -> 100
Época 01/30 | Train CEL: 2.16075 | Valid CEL: 2.16105 | Train Accuracy: 0.37402 | Valid Accuracy: 0.37350
Época 02/30 | Train CEL: 1.92805 | Valid CEL: 1.92961 | Train Accuracy: 0.43418 | Valid Accuracy: 0.43560
Época 03/30 | Train CEL: 1.62223 | Valid CEL: 1.62485 | Train Accuracy: 0.54158 | Valid Accuracy: 0.54310
Época 04/30 | Train CEL: 1.36002 | Valid CEL: 1.36323 | Train Accuracy: 0.63016 | Valid Accuracy: 0.62560
Época 05/30 | Train CEL: 1.17073 | Valid CEL: 1.17557 | Train Accuracy: 0.68436 | Valid Accuracy: 0.67440
Época 06/30 | Train CEL: 1.03942 | Valid CEL: 1.04548 | Train Accur